In [1]:

# Experiment 3: Decision Tree using ID3 Algorithm

import pandas as pd
import math
from collections import Counter

# Step 1: Create Play Tennis dataset

data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
                'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
                'Sunny', 'Overcast', 'Overcast', 'Rain'],

    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool',
                    'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
                    'Mild', 'Mild', 'Hot', 'Mild'],

    'Humidity': ['High', 'High', 'High', 'High', 'Normal',
                 'Normal', 'Normal', 'High', 'Normal', 'Normal',
                 'Normal', 'High', 'Normal', 'High'],

    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak',
             'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
             'Strong', 'Strong', 'Weak', 'Strong'],

    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes',
                   'No', 'Yes', 'No', 'Yes', 'Yes',
                   'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)

print("Training Dataset:")
print(df)

# Step 2: Calculate Entropy

def entropy(target):
    counts = Counter(target)
    total = len(target)

    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result

# Step 3: Calculate Information Gain

def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])

    values = df[attribute].unique()
    weighted_entropy = 0

    for value in values:
        subset = df[df[attribute] == value]
        weight = len(subset) / len(df)

        weighted_entropy += weight * entropy(subset[target])

    return total_entropy - weighted_entropy

# Step 4: Build ID3 Decision Tree

def id3(df, attributes, target):

    # If all examples belong to one class
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no attributes are left
    if not attributes:
        return df[target].mode()[0]

    # Find best attribute
    best_attribute = max(
        attributes,
        key=lambda a: information_gain(df, a, target)
    )

    tree = {best_attribute: {}}

    remaining = [a for a in attributes if a != best_attribute]

    for value in df[best_attribute].unique():

        subset = df[df[best_attribute] == value]

        tree[best_attribute][value] = id3(
            subset, remaining, target
        )

    return tree

# Step 5: Build the decision tree

attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']

tree = id3(df, attributes, 'PlayTennis')

print("\nDecision Tree:")
print(tree)

# Step 6: Classify a new sample

def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    if value not in tree[attribute]:
        return "Unknown"

    return predict(tree[attribute][value], sample)


new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = predict(tree, new_sample)

print("\nNew Sample:")
print(new_sample)

print("\nPredicted Result:", result)


Training Dataset:
     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample:
{'Outlook': 'Sunny